# Measurable Experiment & Sensitivity Analysis
## Shift Workload-Balancing Simulator for Hospital Patient Transfers (Review 1 Target)

This notebook executes the measurable experiment comparing 4 operating scenarios using the synthetic hospital dataset (~5,200 patient records across 3 facilities and 15 units).

In [1]:
import os
import sys
import pandas as pd
import numpy as np

# Ensure project root path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.data_cleaning import run_cleaning_pipeline
from src.simulator import run_scenario_simulation, run_all_operating_scenarios, run_sensitivity_analysis

# 1. Load Cleaned Dataset
df_units, df_nurses, df_patients = run_cleaning_pipeline(raw_dir="../data/raw", clean_dir="../data/clean")
print(f"Loaded {len(df_patients)} patients, {len(df_nurses)} nurses, and {len(df_units)} units.")

## 1. Operating Scenarios Experiment Table

We evaluate 4 operating scenarios:
1. **Baseline**: Normal patient acuity and normal staffing.
2. **High Acuity**: Patient acuity/workload increased by 25%.
3. **Staff Shortage**: Available nurse count reduced by 20%.
4. **Combined Stress**: High Acuity (+25%) + Staff Shortage (-20%).

In [2]:
results, summary_df = run_all_operating_scenarios(df_units, df_nurses, df_patients)

experiment_table = summary_df[[
    "Scenario_Name", "Pre_Staffing_Gap", "Post_Staffing_Gap",
    "Pre_Workload_Imbalance_Std", "Post_Workload_Imbalance_Std",
    "Imbalance_Reduction_Pct", "Safe_Reassignment_Capacity", "Unsafe_Attempts_Prevented"
]].copy()

experiment_table.columns = [
    "Scenario", "Gap Before", "Gap After",
    "Imbalance Std (Before)", "Imbalance Std (After)",
    "Improvement %", "Safe Transfers", "Unsafe Blocked"
]

display(experiment_table)

## 2. Sensitivity Analysis

Varying patient acuity (+10%, +25%, +40%) and staff shortages (-10%, -20%, -30%).

In [3]:
sens_matrix = run_sensitivity_analysis(df_units, df_nurses, df_patients)
display(sens_matrix)

## 3. Error & Failure Analysis

### Key Prototype Failure Modes:
1. **Skill Mismatch**: Reassignment requested for ICU patient when only Med-Surg certified nurses exist.
2. **Donor Unit Depletion**: Donor unit has surplus above workload, but transfer would drop unit below `Minimum_Staff` threshold.
3. **Physical Capacity Bottleneck**: Receiving unit is at physical bed capacity (Occupied = Capacity), preventing patient transfer.
4. **Static Roster Latency**: Roster data updates delayed relative to rapid patient acuity changes.